# Module 3.2 — Dynamic Metadata Extraction (v3.2 — Model Upgrade)

**Thay đổi so với v3.2:**
- 🔄 **Model**: `Qwen2.5-7B-Instruct` → `Phi-4-mini-instruct` (4-bit quantization, ~3GB VRAM, fit T4)
- ✅ **Tại sao Phi-4-mini**: Nhỏ gọn hơn (~3.8B), nhanh hơn, VRAM thấp hơn, vẫn theo instruction JSON tốt
- ✅ **4-bit via bitsandbytes**: VRAM chỉ ~3GB, còn rất nhiều dư cho KV-cache
- ✅ **Giữ nguyên**: JSON prefill trick, auto-detect schema, smart truncation, confidence scoring

| Model | VRAM (float16) | VRAM (4-bit) | Khả năng JSON |
|---|---|---|---|
| Qwen2.5-7B-Instruct | ~14GB | ~5GB | ✅ Tốt |
| **Phi-4-mini-instruct** | ~8GB | **~3GB** ✅ | ✅ Tốt |
| Qwen2.5-3B | ~6GB | — | ⚠️ Trung bình |


In [2]:
# bitsandbytes cần thiết cho 4-bit quantization
!pip install -q pydantic json-repair transformers accelerate tqdm
!pip install -q bitsandbytes>=0.41.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 4.4 MB/s eta 0:00:00


In [3]:
import gc
import json
import logging
import re
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Type

import torch
from json_repair import repair_json
from pydantic import BaseModel, Field
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("UIE")

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
@dataclass
class ExtractionConfig:
    model_name: str        = "microsoft/Phi-4-mini-instruct"  # ← Chuyển sang Phi-4-mini
    max_new_tokens: int    = 384    # Phi-4-mini sinh JSON nhanh và chính xác hơn
    max_input_chars: int   = 2000   # Phi-4-mini xử lý context tốt hơn
    max_input_tokens: int  = 1536
    max_retries: int       = 2
    low_confidence_threshold: float = 0.65
    input_file: str  = "/content/drive/MyDrive/VNDigitize/Inputdata.json"
    output_file: str = "/content/drive/MyDrive/VNDigitize/UIE_Extraction_Results_v3_2.json"
    default_schema: str = "BAN_AN_DAN_SU"

CFG = ExtractionConfig()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Thiết bị: {DEVICE}")
if DEVICE == "cuda":
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(
        f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {total_vram:.1f} GB"
    )
    if total_vram < 12:
        logger.info("T4 phát hiện — sẽ dùng 4-bit quantization (~5GB VRAM)")

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# SCHEMA DEFINITIONS (v3.2.1 - Universal Document Support)
# ══════════════════════════════════════════════════════════════════════════════
from pydantic import BaseModel, Field
from typing import Optional, Dict, Type, List

# 1. Nhóm Văn bản Tòa án (Bản án, Quyết định giải quyết vụ án/khiếu nại...)
class VanBanToaAnSchema(BaseModel):
    loai_van_ban: Optional[str] = Field(default=None, description="VD: Bản án sơ thẩm, Quyết định công nhận thuận tình ly hôn, Quyết định đình chỉ xét xử")
    so_hieu: Optional[str] = Field(default=None, description="VD: 03/2022/DSST, 89/2026/QĐST-HNGĐ")
    ngay_ban_hanh: Optional[str] = Field(default=None, description="VD: 23/11/2022")
    ten_toa_an: Optional[str] = Field(default=None, description="Cơ quan ban hành. VD: Tòa án nhân dân tỉnh BN, Tòa án nhân dân Thành phố Hồ Chí Minh")
    vu_viec_tom_tat: Optional[str] = Field(default=None, description="Nội dung tóm tắt vụ việc. VD: Tranh chấp về kiện đòi tài sản, Xin ly hôn, Áp dụng biện pháp xử lý hành chính")
    tham_phan_chu_toa: Optional[str] = Field(default=None, description="Tên Thẩm phán hoặc Chủ tọa phiên tòa/phiên họp")
    nguyen_don_nguoi_khoi_kien: Optional[str] = Field(default=None, description="Tên Nguyên đơn, Người khởi kiện hoặc Người yêu cầu")
    bi_don_nguoi_bi_kien: Optional[str] = Field(default=None, description="Tên Bị đơn, Người bị kiện, hoặc Bị cáo")

# 2. Nhóm Văn bản Hành chính (Quyết định, Thông báo, Kết luận của UBND, Sở ban ngành...)
class VanBanHanhChinhSchema(BaseModel):
    loai_van_ban: Optional[str] = Field(default=None, description="VD: Quyết định, Thông báo, Kết luận thanh tra")
    so_hieu: Optional[str] = Field(default=None, description="VD: 1059/QĐ-UBND, 03/KL-TT")
    ngay_ban_hanh: Optional[str] = Field(default=None, description="VD: 29 tháng 3 năm 2021")
    co_quan_ban_hanh: Optional[str] = Field(default=None, description="VD: UBND thành phố Đà Lạt, Thanh tra tỉnh Vĩnh Long")
    noi_dung_tom_tat: Optional[str] = Field(default=None, description="Về việc gì. VD: Tạm đình chỉ công tác đối với ông Nguyễn Đình Khoa, Thu hồi Giấy chứng nhận quyền sử dụng đất")
    nguoi_ky: Optional[str] = Field(default=None, description="Tên người ký văn bản ở cuối trang. VD: Tôn Thiện San, Võ Văn Phú")

# 3. Nhóm Biên bản (Ghi lời khai, Đối chất, Vi phạm hành chính, Làm việc, Giao nhận...)
class BienBanSchema(BaseModel):
    ten_bien_ban: Optional[str] = Field(default=None, description="VD: Biên bản ghi lời khai, Biên bản đối chất, Biên bản vi phạm hành chính")
    thoi_gian_lap: Optional[str] = Field(default=None, description="Thời gian lập biên bản. VD: Hồi 14 giờ 30 ngày 02 tháng 12 năm 2013")
    dia_diem_lap: Optional[str] = Field(default=None, description="Nơi lập. VD: Công an quận Gò Vấp, Trụ sở Tòa án nhân dân tỉnh Nghệ An")
    nguoi_chu_tri_lap: Optional[str] = Field(default=None, description="Tên cán bộ, Điều tra viên hoặc người đại diện lập biên bản")
    nguoi_tham_gia_doi_tuong: Optional[str] = Field(default=None, description="Tên những người được lấy lời khai, bị lập biên bản, hoặc các bên thỏa thuận")
    noi_dung_chinh: Optional[str] = Field(default=None, description="Tóm tắt ngắn gọn nội dung biên bản. VD: Ghi lời khai về sự việc đánh nhau, Giao nhận tài liệu chứng cứ")

# 4. Nhóm Cơ quan Điều tra / Công an (Kết luận điều tra, Lệnh, Thông báo tố tụng...)
class VanBanDieuTraSchema(BaseModel):
    loai_van_ban: Optional[str] = Field(default=None, description="VD: Bản kết luận điều tra, Lệnh cấm đi khỏi nơi cư trú, Thông báo")
    so_hieu: Optional[str] = Field(default=None, description="VD: 65/KLĐT-PC03, 01/VKS-P2")
    ngay_ban_hanh: Optional[str] = Field(default=None, description="VD: 03 tháng 11 năm 2021")
    co_quan_ban_hanh: Optional[str] = Field(default=None, description="VD: Cơ quan Cảnh sát điều tra Công an tỉnh Quảng Bình, Viện kiểm sát nhân dân")
    ten_bi_can_doi_tuong: Optional[str] = Field(default=None, description="Họ tên bị can, người bị hại hoặc đối tượng bị điều tra")
    toi_danh_vu_an: Optional[str] = Field(default=None, description="Tội danh hoặc tên vụ án. VD: Vi phạm quy định về đầu tư công trình xây dựng gây hậu quả nghiêm trọng, Trộm cắp tài sản")

# 5. Nhóm Đơn từ, Giấy cam kết, Công văn công dân (Catch-all cho các loại giấy tờ cá nhân tự viết hoặc mẫu đơn)
class DonTuCamKetSchema(BaseModel):
    loai_giay_to: Optional[str] = Field(default=None, description="VD: Đơn tố cáo, Bản cam kết, Thư cảm ơn, Giấy xác nhận đăng ký đất đai")
    nguoi_viet_don_cam_ket: Optional[str] = Field(default=None, description="Tên người đứng đơn hoặc người cam kết. VD: Trương Đừng, Đào Thị Kiều Oanh")
    noi_nhan_co_quan_giai_quyet: Optional[str] = Field(default=None, description="Kính gửi ai/cơ quan nào (nếu có)")
    ngay_thang_nam: Optional[str] = Field(default=None, description="Ngày tháng ghi trên đơn từ/cam kết")
    noi_dung_tom_tat: Optional[str] = Field(default=None, description="Tóm tắt nội dung người dân muốn trình bày, cam kết hoặc tố cáo")

# --- ĐĂNG KÝ SCHEMA & KEYWORDS ---

SCHEMA_REGISTRY: Dict[str, Type[BaseModel]] = {
    "VAN_BAN_TOA_AN": VanBanToaAnSchema,
    "VAN_BAN_DIEU_TRA": VanBanDieuTraSchema,
    "QUYET_DINH_HANH_CHINH": VanBanHanhChinhSchema,
    "BIEN_BAN": BienBanSchema,
    "DON_TU_CAM_KET": DonTuCamKetSchema,
}

# Thứ tự keyword rất quan trọng (ưu tiên kiểm tra từ trên xuống dưới)
SCHEMA_KEYWORDS: List[tuple] = [
    ("VAN_BAN_TOA_AN", ["tòa án nhân dân", "bản án số", "quyết định đình chỉ xét xử", "tòa phúc thẩm", "tòa án", "thẩm phán", "xét xử"]),
    ("VAN_BAN_DIEU_TRA", ["kết luận điều tra", "cơ quan cảnh sát điều tra", "viện kiểm sát", "khởi tố bị can", "lệnh cấm đi khỏi", "truy tố", "tội phạm"]),
    ("BIEN_BAN", ["biên bản vi phạm", "biên bản ghi lời khai", "biên bản đối chất", "biên bản thỏa thuận", "biên bản giao nhận", "hồi", "giờ", "phút", "tại"]),
    ("QUYET_DINH_HANH_CHINH", ["quyết định", "ủy ban nhân dân", "ubnd", "thanh tra tỉnh", "thu hồi", "sở tài nguyên", "đình chỉ công tác"]),
    ("DON_TU_CAM_KET", ["đơn tố cáo", "bản cam kết", "giấy cam kết", "thư cảm ơn", "giấy xác nhận", "đơn khiếu nại", "tôi tên là"]),
]

def auto_detect_schema(text: str) -> str:
    text_lower = text.lower()
    for schema_name, keywords in SCHEMA_KEYWORDS:
        if any(kw.lower() in text_lower for kw in keywords):
            return schema_name
    return "DON_TU_CAM_KET" # Default fallback (phù hợp cho các form tự do)

# logger.info(f"Schema đã đăng ký: {list(SCHEMA_REGISTRY.keys())}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD MODEL — Phi-4-mini-instruct với 4-bit quantization
# VRAM sử dụng: ~3GB (T4 16GB còn rất nhiều dư cho KV-cache)
# ══════════════════════════════════════════════════════════════════════════════

logger.info(f"Đang tải: {CFG.model_name} với 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 tốt nhất cho LLM
    bnb_4bit_compute_dtype=torch.float16,  # compute vẫn dùng fp16
    bnb_4bit_use_double_quant=True,     # double quant: tiết kiệm thêm ~0.4 bit/param
)

tokenizer = AutoTokenizer.from_pretrained(
    CFG.model_name,
    padding_side="left",
)

model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

# Kiểm tra VRAM sau khi load
if DEVICE == "cuda":
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f"✅ Model loaded | VRAM dùng: {used:.1f}/{total:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PROMPT + INFERENCE
# ══════════════════════════════════════════════════════════════════════════════

def smart_truncate(text: str, max_chars: int) -> str:
    """Giữ 80% phần đầu + scan regex lấy dòng key từ phần đuôi."""
    if len(text) <= max_chars:
        return text
    head = text[:int(max_chars * 0.8)]
    tail = text[int(max_chars * 0.8):]
    key_pats = [
        r"(?:bị đơn|nguyên đơn|bị cáo)[:\s].+",
        r"\d{1,3}/\d{4}/[A-Z]{2,}",
        r"ngày\s+\d{1,2}\s+tháng",
    ]
    key_lines = [
        line.strip() for line in tail.splitlines()
        if any(re.search(p, line, re.IGNORECASE) for p in key_pats)
    ]
    snippet = " | ".join(key_lines)[:int(max_chars * 0.2)]
    return head + ("\n...[cắt]...\n" + snippet if snippet else "...[cắt]")


def _schema_to_field_list(schema_class: Type[BaseModel]) -> str:
    props = schema_class.model_json_schema().get("properties", {})
    return "\n".join(f"- {k}: {v.get('description', '')}" for k, v in props.items())


def _empty_template(schema_class: Type[BaseModel]) -> str:
    props = schema_class.model_json_schema().get("properties", {})
    return json.dumps({k: None for k in props}, ensure_ascii=False, indent=2)


def build_primary_prompt(schema_class: Type[BaseModel], raw_text: str) -> list[dict]:
    field_list = _schema_to_field_list(schema_class)
    template   = _empty_template(schema_class)
    truncated  = smart_truncate(raw_text, CFG.max_input_chars)

    system_msg = (
        "Bạn là chuyên gia trích xuất dữ liệu văn bản pháp lý Việt Nam.\n"
        "Nhiệm vụ: Đọc văn bản và trả về JSON với các trường sau.\n"
        "Quy tắc:\n"
        "- Chỉ trả về JSON thuần túy, không markdown, không giải thích.\n"
        "- Không tìm thấy → null (không được bỏ trống chuỗi).\n"
        "- Danh sách nhiều người: dùng '; ' làm dấu phân cách.\n\n"
        f"TRƯỜNG CẦN TRÍCH XUẤT:\n{field_list}\n\n"
        f"TEMPLATE (điền giá trị vào, giữ nguyên key):\n{template}"
    )

    return [
        {"role": "system",    "content": system_msg},
        {"role": "user",      "content": f"VĂN BẢN:\n{truncated}"},
        {"role": "assistant", "content": "{"},  # JSON prefill
    ]


def build_retry_prompt(schema_class: Type[BaseModel], raw_text: str) -> list[dict]:
    fields = list(schema_class.model_json_schema().get("properties", {}).keys())
    return [
        {"role": "system", "content": (
            f"Trích xuất {', '.join(fields)} từ văn bản tiếng Việt.\n"
            f"Chỉ trả về JSON. Không tìm thấy → null."
        )},
        {"role": "user",      "content": smart_truncate(raw_text, 1200)},
        {"role": "assistant", "content": "{"},  # JSON prefill
    ]


def run_inference(messages: list[dict]) -> str:
    """Inference với JSON prefill — hỗ trợ cả transformers cũ lẫn mới."""
    has_prefill = (
        messages[-1]["role"] == "assistant"
        and messages[-1]["content"].startswith("{")
    )

    # Thử continue_final_message (transformers ≥ 4.43)
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=not has_prefill,
            continue_final_message=has_prefill,
        )
    except TypeError:
        # Fallback: bỏ assistant message, thêm '{' thủ công vào cuối template
        msgs = [m for m in messages if m["role"] != "assistant"]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        ) + "{"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=CFG.max_input_tokens,
    ).to(model.device)

    input_len = inputs.input_ids.shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=CFG.max_new_tokens,
            do_sample=False,          # Greedy — nhanh, ổn định
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_ids = output_ids[0][input_len:]
    decoded = tokenizer.decode(new_ids, skip_special_tokens=True)

    # Prefill '{' bị nuốt vào input — prepend lại
    if has_prefill and not decoded.strip().startswith("{"):
        return "{" + decoded
    return decoded

In [ ]:
# ── JSON Parser ───────────────────────────────────────────────────────────────

def clean_and_parse(raw: str, doc_id: str = "") -> dict | None:
    raw = raw.strip()

    # Thử lấy block ```json...```
    m = re.search(r"```(?:json)?\s*([\s\S]+?)```", raw)
    candidate = m.group(1).strip() if m else raw

    # Lấy từ { đến }
    s, e = candidate.find("{"), candidate.rfind("}")
    if s != -1 and e != -1:
        candidate = candidate[s: e + 1]

    # Parse JSON
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # json-repair fallback
    try:
        repaired = repair_json(candidate, return_objects=True)
        if isinstance(repaired, dict) and repaired:
            return repaired
    except Exception:
        pass

    if doc_id:
        logger.warning(f"[{doc_id}] Raw output (300c): {repr(raw[:300])}")
    return None

In [ ]:
# ── Confidence Scoring ────────────────────────────────────────────────────────

_RE_BAN_AN  = re.compile(r"^\d{1,3}/\d{4}/[A-Z]{2,}(-[A-Z]+)?$")
_RE_THU_LY  = re.compile(r"^\d{1,3}/\d{4}/[A-Z]{2,}-[A-Z]+$")
_RE_DATE_VN = re.compile(r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{4}")
_RE_DATE_TXT= re.compile(r"\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}")
_RE_CCCD    = re.compile(r"^\d{9}$|^\d{12}$")
_RE_DIGIT   = re.compile(r"\d")


def compute_field_confidence(value: Any, field_name: str) -> float:
    if value is None or str(value).strip() == "":
        return 0.0
    text  = str(value).strip()
    fname = field_name.lower()

    if "so_ban_an" in fname:
        return 0.95 if _RE_BAN_AN.match(text) else (0.70 if _RE_DIGIT.search(text) else 0.45)

    if "thu_ly" in fname:
        return 0.95 if _RE_THU_LY.match(text) else (0.70 if _RE_DIGIT.search(text) else 0.45)

    if any(k in fname for k in ("ngay", "ngày", "date", "xet_xu")):
        if _RE_DATE_VN.search(text) or _RE_DATE_TXT.search(text):
            return 0.95
        return 0.55 if _RE_DIGIT.search(text) else 0.30

    if any(k in fname for k in ("nam_sinh", "year")):
        try:
            return 0.95 if 1900 <= int(text) <= 2015 else 0.50
        except ValueError:
            return 0.30

    if any(k in fname for k in ("cccd", "cmnd")):
        return 0.95 if _RE_CCCD.match(text.replace(" ", "")) else (0.65 if _RE_DIGIT.search(text) else 0.35)

    name_fields = ("ho_ten", "tham_phan", "thu_ky", "kiem_sat", "ten_bi",
                   "nguyen_don", "bi_don", "hoi_tham", "ten_nguyen")
    if any(k in fname for k in name_fields):
        if ";" in text:
            parts  = [p.strip() for p in text.split(";") if p.strip()]
            valid  = sum(1 for p in parts if len(p.split()) >= 2)
            return round(min(0.70 + 0.08 * valid, 0.95), 2)
        return 0.88 if (len(text.split()) >= 2 and not _RE_DIGIT.search(text)) else 0.70

    if any(k in fname for k in ("toa_an", "loai", "toi_danh", "noi_cap")):
        return 0.85 if len(text.split()) >= 3 else 0.70

    return 0.78 if len(text.split()) >= 2 else 0.60


def _cross_validate_year(scored: dict) -> None:
    ban_an = (scored.get("so_ban_an",   {}) or {}).get("value")
    ngay   = (scored.get("ngay_xet_xu", {}) or {}).get("value")
    if not ban_an or not ngay:
        return
    m1 = re.search(r"/(\d{4})/", str(ban_an))
    m2 = re.search(r"(\d{4})",   str(ngay))
    if m1 and m2 and m1.group(1) != m2.group(1):
        for key in ("so_ban_an", "ngay_xet_xu"):
            if key in scored:
                scored[key]["confidence"] = round(scored[key]["confidence"] * 0.7, 2)
                scored[key]["flagged"]    = True
                scored[key]["warning"]    = "Năm không khớp so_ban_an ↔ ngay_xet_xu"


def score_extraction(extracted: dict, schema_class: Type[BaseModel]) -> tuple[dict, float]:
    scored, all_confs = {}, []
    for fname in schema_class.model_json_schema().get("properties", {}):
        val  = extracted.get(fname)
        conf = compute_field_confidence(val, fname)
        scored[fname] = {"value": val, "confidence": conf, "flagged": conf < CFG.low_confidence_threshold}
        all_confs.append(conf)
    _cross_validate_year(scored)
    return scored, round(sum(all_confs) / len(all_confs), 3) if all_confs else 0.0

In [ ]:
# ── Input Adapter + Extractor ─────────────────────────────────────────────────

def parse_input_doc(doc: dict) -> tuple[str, str, str]:
    """Hỗ trợ cả format cũ (module2_output/document_id) và mới (ground_truth_text/sample_id)."""
    doc_id = (
        doc.get("document_id")
        or doc.get("sample_id")
        or doc.get("image_name", "Unknown")
    )
    if "module2_output" in doc and isinstance(doc["module2_output"], dict):
        raw_text = doc["module2_output"].get("raw_text", "")
    elif "ground_truth_text" in doc:
        raw_text = doc["ground_truth_text"]
    else:
        raw_text = doc.get("text", "")
    return str(doc_id), str(raw_text), doc.get("schema_type", "")


def extract_document(doc: dict) -> dict:
    doc_id, raw_text, schema_hint = parse_input_doc(doc)

    # Chọn schema
    if schema_hint and schema_hint in SCHEMA_REGISTRY:
        schema_name = schema_hint
    else:
        schema_name = auto_detect_schema(raw_text)
        logger.info(f"[{doc_id}] Auto-detect → '{schema_name}'")

    schema_class = SCHEMA_REGISTRY[schema_name]
    parsed, last_raw = None, ""

    for attempt in range(1, CFG.max_retries + 1):
        try:
            messages = (
                build_primary_prompt(schema_class, raw_text) if attempt == 1
                else build_retry_prompt(schema_class, raw_text)
            )
            t0       = time.time()
            last_raw = run_inference(messages)
            elapsed  = time.time() - t0
            parsed   = clean_and_parse(last_raw, doc_id)

            if isinstance(parsed, dict) and parsed:
                logger.info(f"[{doc_id}] ✅ OK lần {attempt} ({elapsed:.1f}s)")
                break
            else:
                parsed = None
                logger.warning(f"[{doc_id}] ⚠️  Parse lỗi lần {attempt} ({elapsed:.1f}s)")

        except torch.cuda.OutOfMemoryError:
            logger.error(f"[{doc_id}] 💥 OOM — giảm context")
            torch.cuda.empty_cache()
            CFG.max_input_chars  = int(CFG.max_input_chars  * 0.6)
            CFG.max_input_tokens = int(CFG.max_input_tokens * 0.6)

        except Exception as e:
            logger.error(f"[{doc_id}] ❌ Lỗi lần {attempt}: {e}")

    if parsed is None:
        return {
            "document_id": doc_id, "schema_used": schema_name,
            "extracted_dynamic_data": None,
            "confidence_overall": 0.0,
            "error": "json_parse_failed",
            "raw_output_sample": last_raw[:300],
        }

    scored, overall = score_extraction(parsed, schema_class)
    return {
        "document_id": doc_id, "schema_used": schema_name,
        "extracted_dynamic_data": scored,
        "confidence_overall": overall,
        "error": None,
    }

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  BƯỚC 1 — Chạy debug 1 document trước để xác nhận model hoạt động
# ══════════════════════════════════════════════════════════════════════════════

with open(CFG.input_file, "r", encoding="utf-8") as f:
    _all_docs = json.load(f)
if isinstance(_all_docs, dict):
    _all_docs = [_all_docs]

_doc_id, _raw, _hint = parse_input_doc(_all_docs[0])
_schema_name  = auto_detect_schema(_raw)
_schema_class = SCHEMA_REGISTRY[_schema_name]
_messages     = build_primary_prompt(_schema_class, _raw)

print(f"Doc ID: {_doc_id}")
print(f"Schema: {_schema_name}")
print(f"Text (400c): {_raw[:400]}")
print("\n--- RUNNING INFERENCE ---")

_raw_out = run_inference(_messages)
print("\n=== RAW MODEL OUTPUT ===")
print(_raw_out[:800])
print("========================")

_parsed = clean_and_parse(_raw_out)
print(f"\n✅ Parsed OK: {_parsed is not None}")
if _parsed:
    print(json.dumps(_parsed, ensure_ascii=False, indent=2))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  BƯỚC 2 — Main pipeline (chạy sau khi debug OK)
# ══════════════════════════════════════════════════════════════════════════════

with open(CFG.input_file, "r", encoding="utf-8") as f:
    documents = json.load(f)
if isinstance(documents, dict):
    documents = [documents]

logger.info(f"Tổng số tài liệu: {len(documents)}")

results = []
t_start = time.time()

for doc in tqdm(documents, desc="Extracting", unit="doc"):
    result = extract_document(doc)
    results.append(result)
    torch.cuda.empty_cache()
    gc.collect()

# ── Thống kê ──────────────────────────────────────────────────────────────────
total_time = time.time() - t_start
success    = [r for r in results if not r.get("error")]
errors     = [r for r in results if r.get("error")]
avg_conf   = sum(r["confidence_overall"] for r in success) / len(success) if success else 0
high_conf  = sum(1 for r in success if r["confidence_overall"] >= 0.80)
med_conf   = sum(1 for r in success if 0.65 <= r["confidence_overall"] < 0.80)
low_conf   = sum(1 for r in success if r["confidence_overall"] < 0.65)

print("\n" + "═"*58)
print(f"Hoàn thành  : {len(results)} doc | {total_time:.1f}s tổng | {total_time/len(results):.1f}s/doc")
print(f"Thành công  : {len(success)}/{len(results)} ({len(success)/len(results):.0%}) | Lỗi: {len(errors)}")
print(f"Confidence  : TB={avg_conf:.2%} | ≥80%: {high_conf} | 65-80%: {med_conf} | <65%: {low_conf}")
print("═"*58)

if errors:
    print("\nDocuments lỗi:")
    for e in errors[:10]:  # Chỉ in 10 cái đầu
        print(f"  [{e['document_id']}] raw='{e.get('raw_output_sample','')[:80]}'")

with open(CFG.output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
logger.info(f"✅ Đã lưu → {CFG.output_file}")